# Módulo 6 — Pronóstico y evaluación de modelos

**Curso: Análisis y Pronóstico de Datos Mineros con Python**

> Partición cronológica, horizonte, intervalos, MAE/RMSE/MAPE, baseline naïve y backtesting walk-forward.

---

### Cómo usar este notebook
1. Ábrelo en **Google Colab** y ejecuta las celdas **de arriba hacia abajo**.
2. Los datos se descargan solos desde el repositorio del curso; no tienes que subir nada.
3. Lee las salidas: cada bloque responde una pregunta concreta, no ejecutes por ejecutar.

In [ ]:
# Librerías base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns          # gráficos estadísticos (preinstalado en Colab)

plt.rcParams['figure.figsize'] = (12, 4)
pd.set_option('display.width', 120)

# --- Datos del curso -------------------------------------------------
# Los CSV viven en el repositorio del curso y se descargan solos.
# Si no hubiera internet, la función pide subir el archivo a mano.
REPO_DATOS = 'https://raw.githubusercontent.com/HishanFarfan/curso-datos-mineros/main/datos'

def cargar_datos(nombre, **kw):
    try:
        return pd.read_csv(f'{REPO_DATOS}/{nombre}', **kw)
    except Exception as e:
        print('No se pudo descargar desde GitHub:', e)
    try:
        from google.colab import files          # Colab: subir a mano
        print(f"Sube '{nombre}':")
        return pd.read_csv(next(iter(files.upload())), **kw)
    except ModuleNotFoundError:
        return pd.read_csv(nombre, **kw)         # local: archivo en el cwd

## 1. Carga de la serie preparada

Partimos de la versión ya limpia y ordenada (`_LIMPIO.csv`). En un flujo real usarías la salida del Módulo 1.

In [ ]:
df = cargar_datos('datos_proceso_planta_LIMPIO.csv', parse_dates=['Fecha'])
df = df.sort_values('Fecha').set_index('Fecha')
df = df.asfreq('h')   # eje horario regular; expone huecos como NaN
df.head()

In [ ]:
serie = df['Recuperacion_pct'].dropna()

## 2. Partición cronológica (nunca aleatoria)

In [ ]:
n_train = int(len(serie) * 0.8)
train, test = serie.iloc[:n_train], serie.iloc[n_train:]
print(len(train), 'train  |', len(test), 'test')
print('corte:', train.index[-1])

## 3. Ajuste sobre train y pronóstico

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
modelo = SARIMAX(train, order=(2,1,1), seasonal_order=(1,0,1,24),
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
pred = modelo.get_forecast(steps=len(test))
fc = pred.predicted_mean
ci = pred.conf_int()

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
train.iloc[-300:].plot(ax=ax, label='train')
test.plot(ax=ax, label='observado')
fc.plot(ax=ax, label='pronóstico')
ax.fill_between(ci.index, ci.iloc[:,0], ci.iloc[:,1], alpha=0.2)
ax.legend(); plt.show()

## 4. Métricas de error

In [ ]:
def metricas(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    mae = np.mean(np.abs(y - yhat))
    rmse = np.sqrt(np.mean((y - yhat)**2))
    mape = 100 * np.mean(np.abs((y - yhat) / y))
    me = np.mean(y - yhat)   # sesgo
    return dict(MAE=mae, RMSE=rmse, MAPE=mape, ME=me)

metricas(test, fc)

## 5. Baselines

In [ ]:
naive = pd.Series(train.iloc[-1], index=test.index)
naive_est = serie.shift(24).iloc[n_train:]
print('naïve      ', {k: round(v,3) for k,v in metricas(test, naive).items()})
print('naïve 24 h ', {k: round(v,3) for k,v in metricas(test.iloc[24:], naive_est.iloc[24:]).items()})
print('SARIMA     ', {k: round(v,3) for k,v in metricas(test, fc).items()})

## 6. Backtesting walk-forward (un paso, ventana expansiva)

In [ ]:
# Reajuste periódico para acotar el costo (cada 24 h)
hist = list(train)
idx = list(train.index)
pasos = test.iloc[:240]           # 10 días de evaluación
preds = []
modelo_wf = None
for i, (t, y) in enumerate(pasos.items()):
    if i % 24 == 0:
        modelo_wf = SARIMAX(pd.Series(hist, index=idx), order=(2,1,1),
                            seasonal_order=(1,0,1,24),
                            enforce_stationarity=False,
                            enforce_invertibility=False).fit(disp=False)
        res_wf = modelo_wf
    else:
        res_wf = modelo_wf.append(pd.Series(hist[-1:], index=idx[-1:]), refit=False)
    preds.append(res_wf.forecast(1).iloc[0])
    hist.append(y); idx.append(t)
preds = pd.Series(preds, index=pasos.index)
metricas(pasos, preds)

In [ ]:
err = (pasos - preds).abs()
err.plot(title='|error| walk-forward'); plt.show()
print('MAE medio backtest:', round(err.mean(), 3))

## 7. Error por horizonte

In [ ]:
h_pred = modelo.get_forecast(steps=48).predicted_mean
obs = test.iloc[:48]
for h in [1, 2, 4, 8, 24, 48]:
    print(f'h={h:>2}  MAE acumulado = {np.mean(np.abs(obs.iloc[:h].values - h_pred.iloc[:h].values)):.3f}')

## 8. Tabla de decisión (complétala)

| Modelo | MAE | RMSE | MAPE | ¿supera naïve? | ¿estable en backtest? |
|---|---|---|---|---|---|
| naïve | | | | — | |
| SARIMA | | | | | |

## Actividades sugeridas

1. Cambia el corte a 70/30 y observa si la conclusión se mantiene.
2. Repite la evaluación para `Tonelaje_tph`.
3. ¿A partir de qué horizonte el MAE supera el 5 % del valor típico de la variable?
4. ¿El modelo tiene sesgo (ME ≠ 0)? ¿Subestima o sobrestima?

---
## Cierre

Sabemos cuánto se equivoca el modelo, a qué horizonte y si supera una referencia trivial. El Módulo 7 usa el error como señal de vigilancia.